# trailrunner in five minutes

One demand, carried end to end: **1000 kg of CO<sub>2</sub> captured from the
air, in Switzerland, in 2030.**

Ordinary practice answers that by looking the process up in a dataset and
multiplying. trailrunner asks a model, and the model answers for the demand it
was actually given, in the place and the year it was given. Everything below
follows from that one change.

Nothing here touches the network. The vocabulary lookups come from the committed
`examples/pyst_cache.json` and `examples/pyst_labels.json`, the background
datasets from `examples/background_pack.parquet`, and the parameters from the
committed parquet files beside this notebook.

In [1]:
import sys
from pathlib import Path

# Run from anywhere: nbconvert starts the kernel in the notebook's directory,
# a human might start it from the repository root.
EXAMPLES = Path.cwd() if (Path.cwd() / "showcase_models.py").exists() else Path.cwd() / "examples"
sys.path.insert(0, str(EXAMPLES))

from trailrunner import Demand, Flow
from trailrunner.models.dac import CO2_CAPTURED
from trailrunner.resolution import PystLabels

DEMAND = Demand(
    flow=Flow(iri=CO2_CAPTURED, location="CH", time=2030), amount=1000.0, unit="kg"
)

# What a flow *is* is an IRI in https://vocab.sentier.dev -- not a free-text
# name. The vocabulary also knows what that concept is called, and those names
# are cached beside this notebook, so every print below reads in English with
# no network and no token.
VOCAB = PystLabels(EXAMPLES / "pyst_labels.json", client=None)


def name(iri: str, width: int | None = None) -> str:
    """The vocabulary's name for a concept, else the IRI's last segment."""
    label = VOCAB.label(iri) or iri.rsplit("/", 1)[-1]
    if width is not None and len(label) > width:
        label = label[: width - 1] + "\u2026"  # a column, not a claim: tree() prints it in full
    return label


print(DEMAND.amount, DEMAND.unit, DEMAND.flow.iri)
print("that IRI is:", name(DEMAND.flow.iri))
print("where:", DEMAND.flow.location, " when:", DEMAND.flow.time)

1000.0 kg https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_2811_21
that IRI is: Carbon dioxide
where: CH  when: 2030


## 1. A process is something you run

A `Model` has one method. It takes a `Demand` and returns a `Result`, answering
three questions at once. What did I make, what do I need, what did I emit.

In [2]:
from trailrunner import LocationHierarchy, ParameterSet
from trailrunner.models import dac
from trailrunner.models.dac import DirectAirCapture

HIERARCHY = LocationHierarchy({"CH": "RER", "FR": "RER", "RER": "GLO"})
dac_params = ParameterSet.from_parquet(EXAMPLES / "dac_params.parquet", hierarchy=HIERARCHY)
plant = DirectAirCapture(params=dac_params)

answer = plant.apply(DEMAND)  # no orchestrator involved: a model is callable on its own

for field in ("production", "technosphere", "biosphere"):
    for exchange in getattr(answer, field):
        flow = exchange.flow
        print(
            f"{field:>13}  {exchange.amount:>8.1f} {exchange.unit:<4} "
            f"{name(flow.iri):<32} @{flow.location}/{flow.time}"
        )
print(f"{'provenance':>13}  {answer.provenance}")

   production    1000.0 kg   Carbon dioxide                   @CH/2030
 technosphere    5000.0 MJ   heat from main producers of heat @CH/2030
 technosphere     400.0 kWh  electricity                      @CH/2030
    biosphere   -1000.0 kg   co2-from-air                     @CH/2030
   provenance  {'location_requested': 'CH', 'location_used': 'CH', 'location_fallback': False, 'time_requested': 2030, 'time_used': 2030, 'time_interpolated': False}


Three lists, three destinations. `production` is checked against the demand that
triggered the run and then dropped. `technosphere` goes back on the queue, and is
where the traversal comes from. `biosphere` accumulates into the inventory.
`provenance` records which parameter row the model read and which fallbacks it
took.

The demand arrives as an argument, so the answer can depend on it. Inside
`DirectAirCapture` the regeneration heat responds to the air the plant is
breathing, because colder and drier air carries less CO<sub>2</sub> and less
water to the sorbent per unit of air moved:

```python
penalty = ambient_penalty(row["temperature"], row["humidity"])
heat = row["heat_demand"] * penalty * demand.amount
```

The same 1000 kg, asked for in four different places and years:

In [3]:
print(f"{'where':>6} {'when':>6} {'degC':>6} {'RH':>6} {'penalty':>9} {'heat [MJ]':>11}")
for location in ("CH", "RER"):
    for year in (2020, 2030):
        air = dac_params.at(location=location, time=year)
        answer = plant.apply(
            Demand(flow=Flow(iri=CO2_CAPTURED, location=location, time=year),
                   amount=1000.0, unit="kg")
        )
        heat = [d for d in answer.technosphere if d.flow.iri == dac.HEAT][0]
        print(
            f"{location:>6} {year:>6} {air['temperature']:>6.1f} {air['humidity']:>6.2f} "
            f"{dac.ambient_penalty(air['temperature'], air['humidity']):>9.3f} "
            f"{heat.amount:>11.1f}"
        )

 where   when   degC     RH   penalty   heat [MJ]
    CH   2020    9.0   0.75     0.995      5970.0
    CH   2030   10.0   0.70     1.000      5000.0
   RER   2020   11.0   0.68     0.996      6573.6
   RER   2030   12.0   0.65     0.995      5472.5


`apply` receives the **full** demanded amount, the whole thing that was asked
for, and nothing downstream rescales what comes back. A model whose response
bends with scale can say so.

A model can also compute nothing at all. Where a process has been measured, the
same `Result` carries metered emissions for that place and that year, read from
the same parquet any other parameter comes from.

## 2. Models find each other through a vocabulary

Every flow is identified by an IRI from the hierarchical
[sentier vocabulary](https://vocab.sentier.dev). A `Demand` for
`.../BONSAI2025.1/fi_1730_9` finds whoever declared that same IRI in `produces`,
with no name matching and no unit guessing in between. That is what lets two
models written by two people compose at all, and it is what the orchestrator uses
to walk outward.

```mermaid
flowchart TB
    D([initial demand]) --> Q[[Queue]]
    Q -->|pop demand| C{{ResolutionChain}}
    C -->|ask model tier: who can offer?| G[(Glossary: available models)]
    G -->|Offer: model + demand| C
    C -->|nobody offers| X[cutoff, with a reason]
    C -->|selected offer| R[Runner]
    R -->|apply demand| M[Model: your code]
    M -->|Result| R
    R -->|Result - technosphere demands| Q
    R -->|Result - biosphere flows| I[(inventory)]
    X --> L[(Log)]
    R --> L
    L --> P([Report])
```

`Orchestrator.calculate` is a `while queue:` and little else. Pop a demand, ask
the chain who can answer it, hand the offer to the `Runner`, push the `Result`'s
technosphere demands back on, write everything to the `Log`. Every seam in that
sentence is an object you can replace, which also makes the loop easy to watch.
Subclass the chain, print each demand it is asked about, and the traversal
narrates itself.

In [4]:
from showcase_models import MODELS  # the same list `trailrunner run --models` loads
from trailrunner import Glossary, Orchestrator
from trailrunner.resolution import ModelProvider, ResolutionChain


class Narrating(ResolutionChain):
    """A chain that says what it was asked. The Orchestrator takes any chain."""

    def offer(self, demand, exclude=()):
        offer = super().offer(demand, exclude=exclude)
        who = type(offer.model).__name__ if offer else "cutoff (nobody offered)"
        print(f"pop {demand.amount:>9.4g} {demand.unit:<4} {name(demand.flow.iri, 32):<32} -> {who}")
        return offer


tier1 = ModelProvider(Glossary(MODELS))
first = Orchestrator(Narrating([tier1])).calculate(DEMAND)

pop      1000 kg   Carbon dioxide                   -> DirectAirCapture
pop      5000 MJ   heat from main producers of heat -> cutoff (nobody offered)
pop       400 kWh  electricity                      -> GridElectricity
pop     8.511 kWh  electricity-natural-gas          -> GasPower
pop      76.6 kWh  electricity-wind                 -> cutoff (nobody offered)
pop     340.4 kWh  electricity-hydro                -> cutoff (nobody offered)
pop     49.42 MJ   Natural gas, liquefied or in th… -> cutoff (nobody offered)


Seven pops, breadth-first, and every pop after the first is a demand some earlier
model returned. The supply chain assembled itself from four registered models and
one starting demand.

The names come from the vocabulary as well. Each concept carries a
`skos:prefLabel`, read here from a committed cache, so the run prints in words.
Two lines still show identifiers, because `electricity-wind` and
`electricity-hydro` are trailrunner's own invented IRIs and the vocabulary has no
concept for them. Beat 3 returns to that.

Three pops found a model. Four found nobody, and those four are in the report
with a reason and a parent.

In [5]:
print(first.summary())
print()
print(first.tree(labels=VOCAB.label))  # the vocabulary's names, where it has one

3 nodes, 2 inventory entries
4 unresolved (no_model_found: 4)
0 proxies
attribution: allocation=none, capital=per_output

1000 kg Carbon dioxide @CH/2030  [model: DirectAirCapture]
  400 kWh electricity @CH/2030  [model: GridElectricity]
    8.51064 kWh electricity-natural-gas @CH/2030  [model: GasPower]
      49.4166 MJ Natural gas, liquefied or in the gaseous state @CH/2030  [cutoff: no_model_found]
    76.5957 kWh electricity-wind @CH/2030  [cutoff: no_model_found]
    340.426 kWh electricity-hydro @CH/2030  [cutoff: no_model_found]
  5000 MJ heat from main producers of heat @CH/2030  [cutoff: no_model_found]


A gap in the supply chain is data in the answer. Every line says how honestly it
was reached, and a cutoff hangs under the node that asked for it. Loops are
bounded the same way, by `max_depth` and `max_nodes`, with `report.truncated`
saying when they bit.

The same walk from the command line, with no notebook involved:

```bash
uv run trailrunner run \
  "https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_2811_21" \
  --amount 1000 --unit kg --location CH --year 2030 \
  --models examples/showcase_models.py
```

## 3. A demand nobody answers is relaxed along the vocabulary

`ResolutionChain` is a list of providers, asked in order, and the first offer
wins. Tier 1 is the models. Every later tier is a concession, and the tier that
made it writes what it conceded into the node's resolution.

**Tier 2 generalises the demand.** Nothing produces `fi_1730_9`, "heat from main
producers of heat". One `skos:broader` step up sits `fi_1730`, "Steam and hot
water", a real BONSAI concept read from the committed cache. A gas CHP registered
at the parent can answer the relaxed demand.

**Tier 3 borrows a dataset.** Given its `Fleet`, `DirectAirCapture` demands each
plant's construction in the year that plant was built, and a construction model
turns that into steel and aluminium taken from the curated background pack.

`GasCHP` and `DacPlantConstruction` below are written in this notebook rather
than shipped, because nothing in the repository produces `fi_1730` and nothing in
it co-produces. Their efficiencies, prices and material intensities are invented.
The `skos:broader` walk, the pack lookup, the completeness flag, the credit
traversal and the construction pulse are the library, and every block of output
below is what it actually printed.

In [6]:
from trailrunner import AttributionSettings, Exchange, Fleet, Model, Property, Result, Settings
from trailrunner.models.dac import DAC_PLANT
from trailrunner.models.electricity import CO2_FOSSIL, ELECTRICITY, NATURAL_GAS

STEAM_AND_HOT_WATER = "https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_1730"
STEEL = "https://vocab.sentier.dev/products/steel-low-alloyed"
ALUMINIUM = "https://vocab.sentier.dev/products/aluminium-primary"


class GasCHP(Model):
    """Gas-fired combined heat and power. Illustrative efficiencies and prices."""

    produces = [STEAM_AND_HOT_WATER]
    supports = frozenset({"economic", "substitution"})  # it co-produces; see beat 5

    heat_efficiency = 0.50
    electrical_efficiency = 0.35
    co2_per_mj_fuel = 0.056  # the same factor examples/gas_power_params.parquet carries
    heat_price = 0.02        # EUR/MJ
    electricity_price = 0.10  # EUR/kWh

    def apply(self, demand):
        here = {"location": demand.flow.location, "time": demand.flow.time}
        fuel = demand.amount / self.heat_efficiency
        power = fuel * self.electrical_efficiency / 3.6
        return Result(
            production=[
                Exchange(flow=demand.flow, amount=demand.amount, unit=demand.unit,
                         properties=(Property("price", demand.amount * self.heat_price, "EUR"),)),
                Exchange(flow=Flow(iri=ELECTRICITY, **here), amount=power, unit="kWh",
                         properties=(Property("price", power * self.electricity_price, "EUR"),)),
            ],
            technosphere=[Demand(flow=Flow(iri=NATURAL_GAS, **here), amount=fuel, unit="MJ")],
            biosphere=[Exchange(flow=Flow(iri=CO2_FOSSIL, **here),
                                amount=fuel * self.co2_per_mj_fuel, unit="kg")],
            provenance={"fuel_mj": fuel},
        )


class DacPlantConstruction(Model):
    """What a capture plant is made of. Illustrative material intensities."""

    produces = [DAC_PLANT]
    supports = frozenset({"none", "economic", "substitution"})

    steel_per_capacity = 3.0      # kg steel per kg/year of capture capacity
    aluminium_per_capacity = 1.5  # kg aluminium, likewise

    def apply(self, demand):
        here = {"location": demand.flow.location, "time": demand.flow.time}
        return Result(
            production=[Exchange(flow=demand.flow, amount=demand.amount, unit=demand.unit)],
            technosphere=[
                Demand(flow=Flow(iri=STEEL, **here),
                       amount=demand.amount * self.steel_per_capacity, unit="kg"),
                Demand(flow=Flow(iri=ALUMINIUM, **here),
                       amount=demand.amount * self.aluminium_per_capacity, unit="kg"),
            ],
            biosphere=[],
        )

In [7]:
from trailrunner.resolution import (
    BackgroundPack, BackgroundProvider, GeneralisingProvider, PystTaxonomy,
)

# The plants that were actually built: the fleet examples/dac.ipynb demonstrates.
FLEET_ROWS = [
    {"plant": "ch-pilot", "location": "CH", "build_year": 2007, "capacity": 5000.0, "lifetime": 20.0},
    {"plant": "ch-1", "location": "CH", "build_year": 2026, "capacity": 12000.0, "lifetime": 20.0},
    {"plant": "ch-2", "location": "CH", "build_year": 2029, "capacity": 40000.0, "lifetime": 20.0},
]
fleet = Fleet(FLEET_ROWS, units={"capacity": "kg/year", "lifetime": "year"}, hierarchy=HIERARCHY)

MODELS_PLUS = [
    DirectAirCapture(params=dac_params, fleet=fleet),
    *(model for model in MODELS if not isinstance(model, DirectAirCapture)),
    GasCHP(),
    DacPlantConstruction(),
]

tier1 = ModelProvider(Glossary(MODELS_PLUS))
# client=None: no network, ever. Every skos:broader answer comes from the file.
taxonomy = PystTaxonomy(EXAMPLES / "pyst_cache.json", client=None)
tier2 = GeneralisingProvider(tier1, hierarchy=HIERARCHY, taxonomy=taxonomy)
pack = BackgroundPack.from_parquet(EXAMPLES / "background_pack.parquet", hierarchy=HIERARCHY)
CHAIN = ResolutionChain([tier1, tier2, BackgroundProvider(pack)])


def walk(allocation):
    settings = Settings(attribution=AttributionSettings(allocation=allocation))
    return Orchestrator(CHAIN, settings=settings).calculate(DEMAND)


report = walk("economic")  # the CHP co-produces, so the run must state a rule: beat 5
print(report.summary())
print()
print(report.tree(labels=VOCAB.label))

10 nodes, 6 inventory entries
4 unresolved (generalisation_exhausted: 4)
5 proxies (4 incomplete)
attribution: allocation=economic, capital=per_output

1000 kg Carbon dioxide @CH/2030  [model: DirectAirCapture]
  5000 MJ heat from main producers of heat @CH/2030  [proxy: product: fi_1730_9 -> fi_1730]
    5070.42 MJ Natural gas, liquefied or in the gaseous state @CH/2030  [cutoff: generalisation_exhausted]
  400 kWh electricity @CH/2030  [model: GridElectricity]
    8.51064 kWh electricity-natural-gas @CH/2030  [model: GasPower]
      49.4166 MJ Natural gas, liquefied or in the gaseous state @CH/2030  [cutoff: generalisation_exhausted]
    76.5957 kWh electricity-wind @CH/2030  [cutoff: generalisation_exhausted]
    340.426 kWh electricity-hydro @CH/2030  [cutoff: generalisation_exhausted]
  11.5385 kg/year direct-air-capture-plant @CH/2026  [model: DacPlantConstruction]
    34.6154 kg steel-low-alloyed @CH/2026  [background: unit_process, incomplete]
    17.3077 kg aluminium-primary @

In [8]:
heat_node = [node for node in report.nodes if node.demand.flow.iri == dac.HEAT][0]
for key, value in report.proxies[heat_node.id].items():
    print(f"{key:>12}: {value}")

print()
print("   asked, in words:", name(dac.HEAT))
print("answered, in words:", name(STEAM_AND_HOT_WATER))

       model: GasCHP
 relaxations: ['product: fi_1730_9 -> fi_1730']
       asked: https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_1730_9 @CH/2030
    answered: https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_1730 @CH/2030
        tier: generalising

   asked, in words: heat from main producers of heat
answered, in words: Steam and hot water


Ten nodes now, and the tag on each line says which tier put it there. The
borrowed rows carry `incomplete` because the pack holds each dataset's direct
exchanges only, so their own upstream is missing and the report says so.

Two limits worth stating. `direct-air-capture-plant`, `electricity-wind`,
`electricity-hydro` and `electricity-natural-gas` are trailrunner's own IRIs, so
the vocabulary answers 404 for them, they print without a name, and the product
dimension can never relax them. And the background pack holds no electricity
dataset on purpose, since a grid-mix unit process delegates its combustion
upstream and borrowing one would answer a kilowatt hour with a plausible looking
near-zero.

Every concession is deliberate, ordered by the practitioner, and written down.

In [9]:
from trailrunner import viz
from trailrunner.assessment import Method, assess

# IPCC AR6 GWP100, stated here rather than read from a background database:
# the mapping from a gas to its warming potential is a fact about the gas.
GWP100 = Method(
    rows=[
        {"flow_iri": "https://vocab.sentier.dev/flows/co2-fossil", "flow_unit": "kg",
         "location": "GLO", "cf": 1.0},
        {"flow_iri": "https://vocab.sentier.dev/flows/ch4-fossil", "flow_unit": "kg",
         "location": "GLO", "cf": 29.8},
        {"flow_iri": "https://vocab.sentier.dev/flows/n2o", "flow_unit": "kg",
         "location": "GLO", "cf": 273.0},
        # The DAC model's uptake flow. The amount is already negative.
        {"flow_iri": dac.CO2_AIR, "flow_unit": "kg", "location": "GLO", "cf": 1.0},
    ],
    unit="kg CO2-eq",
    name="IPCC AR6 GWP100",
    hierarchy=HIERARCHY,
)

assessment = assess(report, GWP100)
sankey_figure = viz.sankey(report, assessment=assessment)
sankey_figure

## 4. The judgement calls stay with the practitioner

The CHP makes heat and electricity. How its burden splits between them is a
choice, and trailrunner will not make it for you. The refusal happens in the
`Runner`, between applying the model and validating what came back, so the model
neither makes the choice nor sees it.

In [10]:
from trailrunner import UnallocatedCoProduction

try:
    walk("none")
except UnallocatedCoProduction as refusal:
    print("allocation='none' ->", refusal)

allocation='none' -> GasCHP returned co-products (https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_17100) but the run's allocation rule is 'none'; model it monofunctionally or choose a rule


In [11]:
runs = {rule: walk(rule) for rule in ("economic", "substitution")}

for rule, run in runs.items():
    score = assess(run, GWP100).score
    print(f"{rule:>13}: {score:>9.1f} kg CO2-eq   ({run.summary().splitlines()[1]})")

credited = runs["substitution"]
chp_node = [node for node in credited.nodes if node.demand.flow.iri == dac.HEAT][0]
print()
print("what the CHP node recorded under substitution:")
for key, value in credited.attribution[chp_node.id].items():
    print(f"  {key}: {value}")

     economic:    -580.6 kg CO2-eq   (4 unresolved (generalisation_exhausted: 4))
 substitution:    -311.3 kg CO2-eq   (7 unresolved (generalisation_exhausted: 7, of which 3 on a credit branch))

what the CHP node recorded under substitution:
  allocation: substitution
  share: 1.0
  substituted: ['https://vocab.sentier.dev/products/bonsai/2025.1/BONSAI2025.1/fi_17100']


Same model, same 1000 kg, and 581 kg of CO<sub>2</sub>-eq removed under one rule
against 311 kg under the other. Under `substitution` the credit goes back on the
queue as a negative demand, is answered by someone other than the CHP, and has
its own cutoffs counted separately.

The gap between the two numbers is set by `GasCHP`'s invented prices, since
economic allocation partitions by revenue. What the run demonstrates is that the
rule moves the answer and that the report records which rule ran.

## 5. Time rides along

Nothing in the loop was ever told about time. A `Flow` carries its year the way it
carries its location, so every demand pushed, every emission accumulated and
every node logged is already dated. The plants doing the capturing were built in
2026 and 2029. The capture, and the gas heat driving it, happen in 2030.

So the inventory is a time series, and can be characterized as one.

In [12]:
from trailrunner.assessment import assess_dynamic

# No characterization table is passed: default_functions() maps the DAC uptake
# flow to the ordinary CO2 function, because the model emits it as an already
# negative CO2 exchange and nothing should negate it a second time.
dynamic = assess_dynamic(report, metric="radiative_forcing", horizon=100)
print(dynamic.summary())

by_year = dynamic.series.groupby(dynamic.series["date"].dt.year)["amount"].sum()
print()
print("marginal radiative forcing, first years [W/m2]:")
print(by_year.head(6).to_string())

-5.14244e-11 W·yr/m2
metric: radiative_forcing, horizon: 100 years
horizon anchored at: 2026-01-01
0 uncharacterized exchanges
0 wrong unit exchanges
0 undated exchanges
0 beyond-horizon exchanges
4 unresolved
5 proxies

marginal radiative forcing, first years [W/m2]:
date
2027    5.054518e-14
2028    9.235309e-14
2029    4.282174e-14
2030    1.684839e-13
2031   -9.756895e-13
2032   -1.776486e-12


In [13]:
curve_figure = viz.curve(dynamic)
curve_figure

The faint bars are the per-year forcing and the red line is its running total. It
starts above zero, where the plants were built, and turns down once the capture
lands in 2030. Four warming years, then a century of payback, because of *when*
each kilogram happened as much as how much of it there was. A static score gives
one number for all of that.

No matrix was rebuilt and no second model was written. Characterization is a
separate reading of an inventory whose dates were never lost.

## 6. The run leaves a record

In [14]:
import pyarrow.parquet as pq

print(report.summary())

out = Path("showcase_log.parquet")
report.log.to_parquet(out)
table = pq.read_table(out)
print()
print(f"wrote {out.name}: {table.num_rows} rows, {table.num_columns} columns")
print("kinds:", sorted(set(table.column("kind").to_pylist())))
out.unlink()

10 nodes, 6 inventory entries
4 unresolved (generalisation_exhausted: 4)
5 proxies (4 incomplete)
attribution: allocation=economic, capital=per_output

wrote showcase_log.parquet: 142 rows, 19 columns
kinds: ['attribution', 'biosphere', 'node', 'provenance', 'resolution', 'unresolved']


In [15]:
contributions_figure = viz.contributions(
    assessment, by="node", labels={node.id: node.model for node in report.nodes}
)
contributions_figure

Every node, every cutoff, every parameter fallback, every proxy and the rule that
made the number, one row each. Parameters arrive as parquet and the whole run
leaves as parquet, so two studies can be diffed with a single read.

## What this changes

- **A process can depend on its demand.** Location, year, scale and ambient
  conditions live in the model, where a physical dependency belongs.
- **A model can be a measurement.** Metered data for one place and year enters
  the same way computed data does.
- **The supply chain assembles itself.** Models declare vocabulary IRIs, and the
  orchestrator finds who answers what.
- **Missing data is visible.** Cutoffs carry a reason and a position in the chain.
- **Concessions are declared and recorded.** A generalised demand or a borrowed
  dataset is tagged at the node, with what was asked and what answered it.
- **Normative choices are the study's.** A co-producing model waits for the
  allocation rule, and the rule travels with the result.
- **Inventories are time-explicit by construction.** Dates survive the traversal,
  so dynamic characterization needs no second model.
- **The run is a file.** One parquet holds the graph, the gaps and the choices.

---

- The deeper worked example: [`examples/dac.ipynb`](dac.ipynb)